# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ikramkhan-gif1/FlyRank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
# ML-09 setup

import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

REPO_PATH = "/content/FlyRank-ML-Internship"

if not os.path.exists(REPO_PATH):
    !git clone https://github.com/Ikramkhan-gif1/FlyRank-ML-Internship.git

DATA_PATH = os.path.join(
    REPO_PATH,
    "data",
    "raw",
    "content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("Loaded successfully.")


Dataset shape: (30000, 44)
Loaded successfully.


In [14]:
# Recreate the Week-5 target

df["impression_change_pct"] = (
    (
        df["impressions_last_30d"]
        - df["impressions_prev_30d"]
    )
    / df["impressions_prev_30d"].replace(0, np.nan)
) * 100

df["is_declining_label"] = (
    df["impression_change_pct"] <= -20
).astype(int)

print("Target created successfully.")

print("\nTarget distribution:")
print(df["is_declining_label"].value_counts())

print("\nTarget proportions:")
print(
    df["is_declining_label"]
    .value_counts(normalize=True)
    .round(4)
)

Target created successfully.

Target distribution:
is_declining_label
1    16305
0    13695
Name: count, dtype: int64

Target proportions:
is_declining_label
1    0.5435
0    0.4565
Name: proportion, dtype: float64


## 1. Two paper findings + my methodology questions

### Finding 1 — Freshness and performance

The paper reports that freshness is associated with different growth and decline patterns. In particular, pages updated 31–90 days ago showed a stronger growth-to-decline ratio, while the paper also reports a large impression lift for refreshed older pages.

**My methodology question:** How is the “refreshed” versus “stale” label defined, and are the compared page groups sufficiently similar before the refresh? A useful review question is whether pages selected for refreshing already had characteristics that made them more likely to improve. The reported held-out comparisons provide useful evidence, but the result should still be interpreted as an observed association rather than proof that refreshing alone caused the improvement.

### Finding 2 — Content lifecycle and declining pages

The paper reports that growing pages were younger on average than declining pages, while average word count was nearly the same between the two groups.

**My methodology question:** How exactly is the growing-versus-declining label constructed, and does the validation design prevent information from the same page or time period from appearing on both sides of the comparison? A time-aware or grouped validation design would make it easier to assess whether the observed relationship generalizes beyond the pages and period used to calculate the finding.

### Overall audit perspective

These are constructive methodology questions rather than objections to the findings. The paper provides useful observed patterns, but the strength of a claim depends on how the labels are defined, how the comparison groups are constructed, and whether the validation design supports the level of generalization being made.


In [15]:
# Paper finding audit: basic checks on the methodology language

paper_audit_checks = pd.DataFrame({
    "Finding": [
        "Freshness and performance",
        "Content lifecycle"
    ],
    "Question 1": [
        "How is refreshed vs stale defined?",
        "How is growing vs declining defined?"
    ],
    "Question 2": [
        "Are comparison groups sufficiently comparable?",
        "Does validation prevent page/time overlap?"
    ],
    "Safe interpretation": [
        "Observed association; not causal proof",
        "Observed pattern; generalization depends on split design"
    ]
})

paper_audit_checks

,Finding,Question 1,Question 2,Safe interpretation
0,Freshness and performance,How is refreshed vs stale defined?,Are comparison groups sufficiently comparable?,Observed association; not causal proof
1,Content lifecycle,How is growing vs declining defined?,Does validation prevent page/time overlap?,Observed pattern; generalization depends on sp...


## 2. My model under an honest split (before/after)



### Before: ordinary random split

As a comparison point, I first evaluate the Week-5 Logistic Regression model using an ordinary random train/test split. This provides a useful baseline for understanding how performance changes when observations from the same clients can appear in both training and testing data.

### After: client-grouped split

I then re-run the same model using a client-grouped split. Each client is kept entirely in either the training or testing set. This is a more conservative validation design for the question of generalization to unseen clients because it prevents the same client from appearing on both sides of the evaluation.

The comparison is based on the same target, feature set, model type, and ROC-AUC metric. The purpose is to measure how much the evaluation result changes when the split better matches the generalization question.

The grouped result is treated as the more honest estimate for unseen-client generalization.


In [16]:
# Week-5 feature set

features = [
    "search_volume",
    "impressions_90d",
    "days_with_impressions",
    "impressions_last_30d",
    "impressions_prev_30d",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

target = "is_declining_label"

print("Features:")
print(features)

print("\nTarget:")
print(target)

Features:
['search_volume', 'impressions_90d', 'days_with_impressions', 'impressions_last_30d', 'impressions_prev_30d', 'days_since_last_update', 'ctr', 'avg_position']

Target:
is_declining_label


In [17]:
# BEFORE: ordinary random train/test split

random_train, random_test = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df[target]
)

X_train_random = random_train[features].copy()
X_test_random = random_test[features].copy()

y_train_random = random_train[target].copy()
y_test_random = random_test[target].copy()

random_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

random_model.fit(X_train_random, y_train_random)

random_proba = random_model.predict_proba(X_test_random)[:, 1]

random_auc = roc_auc_score(
    y_test_random,
    random_proba
)

print("BEFORE — Random split ROC-AUC:", round(random_auc, 4))
print("Training rows:", len(random_train))
print("Testing rows:", len(random_test))

BEFORE — Random split ROC-AUC: 0.9142
Training rows: 24000
Testing rows: 6000


In [18]:
# AFTER: client-grouped train/test split

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

group_train_idx, group_test_idx = next(
    group_splitter.split(
        df,
        df[target],
        groups=df["client_id"]
    )
)

group_train = df.iloc[group_train_idx].copy()
group_test = df.iloc[group_test_idx].copy()

X_train_grouped = group_train[features].copy()
X_test_grouped = group_test[features].copy()

y_train_grouped = group_train[target].copy()
y_test_grouped = group_test[target].copy()

grouped_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

grouped_model.fit(
    X_train_grouped,
    y_train_grouped
)

grouped_proba = grouped_model.predict_proba(
    X_test_grouped
)[:, 1]

grouped_auc = roc_auc_score(
    y_test_grouped,
    grouped_proba
)

shared_clients = (
    set(group_train["client_id"])
    & set(group_test["client_id"])
)

print("AFTER — Client-grouped ROC-AUC:", round(grouped_auc, 4))
print("Training rows:", len(group_train))
print("Testing rows:", len(group_test))
print("Training clients:", group_train["client_id"].nunique())
print("Testing clients:", group_test["client_id"].nunique())
print("Shared clients:", len(shared_clients))

AFTER — Client-grouped ROC-AUC: 0.8531
Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Shared clients: 0


In [19]:
# Before/after validation comparison

validation_comparison = pd.DataFrame({
    "Validation design": [
        "Random split — BEFORE",
        "Client-grouped split — AFTER"
    ],
    "ROC-AUC": [
        random_auc,
        grouped_auc
    ]
})

validation_comparison["ROC-AUC"] = (
    validation_comparison["ROC-AUC"].round(4)
)

validation_comparison

,Validation design,ROC-AUC
0,Random split — BEFORE,0.9142
1,Client-grouped split — AFTER,0.8531


### Validation comparison

The ordinary random split produced a measured ROC-AUC of **[random AUC]**, while the client-grouped split produced a measured ROC-AUC of **[grouped AUC]**.

The client-grouped result is the more relevant measurement for unseen-client generalization because no client appears in both the training and testing sets.

The difference between the two measurements is an observed validation effect. The random-split result should not automatically be interpreted as better real-world performance because observations from the same clients can appear in both training and testing data.

For this audit, the client-grouped result is therefore treated as the more conservative decision-support measurement for generalization to unseen clients.


### Real failure examples

The following examples show individual test observations where the model's predicted class differed from the observed `is_declining_label`.

These examples are used for error inspection rather than as proof that the model is reliable for every content item. The purpose is to understand where the measured model signal disagrees with the observed target.


In [20]:
# Inspect real failure examples from the grouped test set
# Client identifiers are intentionally excluded from the displayed output.

failure_examples = group_test.copy()

failure_examples["predicted_probability"] = grouped_proba

failure_examples["predicted_label"] = (
    failure_examples["predicted_probability"] >= 0.50
).astype(int)

failure_examples["error_type"] = np.where(
    (
        (failure_examples[target] == 0)
        & (failure_examples["predicted_label"] == 1)
    ),
    "False Positive",
    np.where(
        (
            (failure_examples[target] == 1)
            & (failure_examples["predicted_label"] == 0)
        ),
        "False Negative",
        "Correct"
    )
)

errors = failure_examples[
    failure_examples["error_type"] != "Correct"
].copy()

error_columns = [
    "report_date",
    "search_volume",
    "impressions_90d",
    "days_with_impressions",
    "impressions_last_30d",
    "impressions_prev_30d",
    "days_since_last_update",
    "ctr",
    "avg_position",
    target,
    "predicted_probability",
    "predicted_label",
    "error_type"
]

errors[
    [col for col in error_columns if col in errors.columns]
].head(10)

,search_volume,impressions_90d,days_with_impressions,impressions_last_30d,impressions_prev_30d,days_since_last_update,ctr,avg_position,is_declining_label,predicted_probability,predicted_label,error_type
13,10.0,307,69,85,77,103,0.00,39.8,0,0.537750,1,False Positive
25,70.0,27,18,9,14,20,0.00,7.2,1,0.404665,0,False Negative
36,0.0,371,82,77,95,20,1.35,5.4,0,0.598922,1,False Positive
39,90.0,4,2,0,4,104,0.00,36.3,1,0.372627,0,False Negative
43,0.0,184,16,3,4,20,0.00,2.9,1,0.406114,0,False Negative
44,0.0,64,21,7,51,20,0.00,55.8,1,0.382393,0,False Negative
47,0.0,8,6,1,6,20,0.00,25.5,1,0.341759,0,False Negative
49,0.0,9,6,0,1,8,0.00,10.1,1,0.352957,0,False Negative
51,0.0,2,2,0,1,8,0.00,7.5,1,0.345041,0,False Negative
58,140.0,71,37,5,17,20,0.00,9.1,1,0.473121,0,False Negative


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

I reviewed the final feature set against the target construction and the timing of the measurements.

The target `is_declining_label` is constructed from the change between `impressions_last_30d` and `impressions_prev_30d`. Both of these variables are also included as model features.

This creates target-construction overlap because the model receives measurements that directly determine the label. Therefore, the measured ROC-AUC should not be interpreted as evidence that the model can reliably predict a genuinely future decline.

The target column itself and the derived `impression_change_pct` column are not included directly as model features. However, the two impression-change inputs remain an important limitation.

For this reason, the current model is best described as a measured classification exercise on the defined proxy label, rather than proof of future decline prediction.


In [21]:
# Leakage audit

feature_set = set(features)

direct_target_columns = {
    "is_declining_label",
    "impression_change_pct"
}

label_construction_columns = {
    "impressions_last_30d",
    "impressions_prev_30d"
}

leakage_audit = pd.DataFrame({
    "Feature": features,
    "Used_to_construct_label": [
        feature in label_construction_columns
        for feature in features
    ],
    "Direct_target_column": [
        feature in direct_target_columns
        for feature in features
    ]
})

leakage_audit


,Feature,Used_to_construct_label,Direct_target_column
0,search_volume,False,False
1,impressions_90d,False,False
2,days_with_impressions,False,False
3,impressions_last_30d,True,False
4,impressions_prev_30d,True,False
5,days_since_last_update,False,False
6,ctr,False,False
7,avg_position,False,False


In [22]:
# Check that the target itself and the derived label column
# are not directly included as model features.

print(
    "Target column in features:",
    target in feature_set
)

print(
    "Derived impression_change_pct in features:",
    "impression_change_pct" in feature_set
)

Target column in features: False
Derived impression_change_pct in features: False


In [23]:
# Timing-related audit

timing_columns = [
    "report_date",
    "days_since_last_update",
    "impressions_last_30d",
    "impressions_prev_30d"
]

timing_audit = pd.DataFrame({
    "Column": timing_columns,
    "Exists_in_dataset": [
        column in df.columns
        for column in timing_columns
    ],
    "Used_as_model_feature": [
        column in features
        for column in timing_columns
    ]
})

timing_audit

,Column,Exists_in_dataset,Used_as_model_feature
0,report_date,False,False
1,days_since_last_update,True,True
2,impressions_last_30d,True,True
3,impressions_prev_30d,True,True


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

“My Logistic Regression model predicts whether content will decline and can be used to identify content that should be refreshed.”

### Safer rewritten claim

“On the evaluated dataset, the Logistic Regression model produced a measured ROC-AUC under both random and client-grouped validation. The results provide directional decision-support signal for the defined `is_declining_label`, but the current label is constructed from the same impression measurements used as model features. Therefore, the experiment should be treated as an observed classification result rather than evidence that the model can reliably predict future content decline or that refreshing a page will cause improvement.”

### Why I changed the claim

The original wording goes further than the validation and label construction support. The revised wording uses observed and measured results, describes the signal as directional decision-support, and explicitly states the limitation identified by the leakage audit.


In [24]:
# Evidence supporting the final claim

claim_evidence = pd.DataFrame({
    "Evidence": [
        "Random-split ROC-AUC",
        "Client-grouped ROC-AUC",
        "Shared clients in grouped split",
        "Features directly used in label construction"
    ],
    "Measured result": [
        round(random_auc, 4),
        round(grouped_auc, 4),
        len(shared_clients),
        "impressions_last_30d + impressions_prev_30d"
    ]
})

claim_evidence


,Evidence,Measured result
0,Random-split ROC-AUC,0.9142
1,Client-grouped ROC-AUC,0.8531
2,Shared clients in grouped split,0
3,Features directly used in label construction,impressions_last_30d + impressions_prev_30d


### Final audit note

This notebook compares random and client-grouped validation, checks for target-construction overlap, inspects real model errors, and limits the final claim to what the measured experiment supports.

The main limitation identified is that the target is constructed from impression-change measurements that are also included as model features. Therefore, the ROC-AUC result is treated as a directional decision-support measurement for the defined proxy label rather than as evidence of reliable future prediction.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.